# Native Shor profiling: streaming versus in-process

Use the repository's Python environment with `qdk[qre]`, a C compiler, and GMP development headers/library. First run `cargo build --release -p qep_interop` from the repository root, as described in [the interop README](../../qep_interop/README.md#shors-algorithm-c-and-fortran).

Both runs use two random 50-bit primes and generator 2. Their products may have 99 or 100 bits, and independent runs need not emit identical operation counts. Counting-profiler measurements are bogus and factorization results are ignored. Estimated quantum runtime is zero because these counting profilers assign zero duration to every gate; estimation time below is measured wall-clock time, including compilation and startup.

In [1]:
from pathlib import Path
from time import perf_counter

from qdk.qre import ProfilerResult
from qdk.qre.application import QspApplication
from qdk.qre.profiler import QREProfiler

repository = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "source/qep_interop/examples/shor.c").is_file()
)
examples = repository / "source/qep_interop/examples"
app = QspApplication(
    examples / "shor.c",
    extra_sources=[examples / "arithmetic.c"],
    args=["--run-random", "100"],
    compiler_args=["-Wall", "-Wextra", "-Werror", "-lgmp"],
)


def report(label: str, result: ProfilerResult, elapsed: float) -> None:
    """Report comparable resource counts and end-to-end profiling speed."""
    totals = result.totals
    count = totals.total_operations_count
    if count == 0:
        raise ValueError("The profile contains no counted operations")
    print(label)
    print(f"Qubits: {totals.logical_qubits:,} logical, {totals.physical_qubits:,} physical")
    print(f"Estimated runtime: {totals.total_runtime_sec:.6f} s")
    print(f"Operations: {count:,}")
    print(f"Estimation time: {elapsed:.3f} s")
    print(f"Speed: {elapsed * 1e9 / count:.1f} ns/op ({count / elapsed / 1e6:.3f} million ops/s)")


profiler = QREProfiler.counting()
started = perf_counter()
streaming_result = profiler.profile(app)
streaming_elapsed = perf_counter() - started
report("Streaming counting profiler (compilation, startup, and pipe transport)", streaming_result, streaming_elapsed)

Running 407323425456803850885381427291 = 636656237132467 * 639785494431673 (50-bit primes), generator=2
Estimated frequency=219511661769333129847279447521401049376754469031376282445991/803469022129495137770981046170581301261101496891396417650688
Estimated period=406511412373166478915497043485
Shot time: 62.874624114 s
Streaming counting profiler (compilation, startup, and pipe transport)
Qubits: 301 logical, 301 physical
Estimated runtime: 0.000000 s
Operations: 123,924,540
Estimation time: 63.370 s
Speed: 511.4 ns/op (1.956 million ops/s)


In [2]:
import subprocess

app.args = ["--profile", "100"]
profile_path = repository / "source/qep/profile_viewer/profiles/shor_c_100.json"
profile_path.parent.mkdir(parents=True, exist_ok=True)
started = perf_counter()
executable = app.compile()
subprocess.run([str(executable), *app.args], cwd=repository, check=True)
in_process_result = ProfilerResult.from_json(profile_path.read_text(encoding="utf-8"))
in_process_elapsed = perf_counter() - started
report("In-process counting profiler (compilation, startup, and profile I/O)", in_process_result, in_process_elapsed)

Profile saved to source/qep/profile_viewer/profiles/shor_c_100.json
Profiling 1025722884991348209563541845527 = 968227202837017 * 1059382427994031 (50-bit primes), generator=2
Estimated frequency=2989683869855514622234909145283410152180545247927934337834666/3213876088517980551083924184682325205044405987565585670602752
Estimated period=294215543258304515993756998141
Profiling measurements are bogus; the factorization result is ignored.
Total operations count: 128864187
Profiling time: 3.633293708 s
Profiling speed: 28.195 ns/op
In-process counting profiler (compilation, startup, and profile I/O)
Qubits: 304 logical, 304 physical
Estimated runtime: 0.000000 s
Operations: 128,864,187
Estimation time: 4.063 s
Speed: 31.5 ns/op (31.716 million ops/s)
